# Compute Author Embeddings for AER

Computes unweighted mean of work embeddings per author cluster.
Used as a content-similarity signal in the AER pipeline for merge/split detection.

- **Source**: `work_embeddings_v2` (414M work embeddings, 1536-dim)
- **Method**: Unweighted mean of all work embeddings per author
- **Output**: One 1536-dim embedding per author in Databricks (not ES)
- **Job**: #105.6 aer-author-embeddings

In [ ]:
# Configuration
OUTPUT_TABLE = "openalex.aer.author_embeddings"
EMBEDDINGS_TABLE = "openalex.vector_search.work_embeddings_v2"
WORKS_TABLE = "openalex.works.openalex_works"
EMBEDDING_DIM = 1536
BATCH_SIZE = 1_000_000  # authors per batch

## Setup output table

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS aer")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {OUTPUT_TABLE} (
        author_id BIGINT,
        work_count INT,
        embedding ARRAY<DOUBLE>
    )
""")

existing = spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']
print(f"Output table has {existing:,} authors already computed")

## Build batch list

Get all distinct author IDs that have work embeddings but aren't yet in the output table.
Split into batches by ID range.

In [ ]:
# Get author IDs that still need computing
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW remaining_authors AS
    SELECT DISTINCT CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id
    FROM {WORKS_TABLE} w
    LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
    WHERE a.author.id IS NOT NULL
      AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) NOT IN (
          SELECT author_id FROM {OUTPUT_TABLE}
      )
""")

remaining = spark.sql("SELECT COUNT(*) AS n FROM remaining_authors").collect()[0]['n']
print(f"Authors remaining: {remaining:,}")

if remaining == 0:
    print("Nothing to do!")
else:
    # Get ID range for batching
    bounds = spark.sql("SELECT MIN(author_id) AS lo, MAX(author_id) AS hi FROM remaining_authors").collect()[0]
    print(f"ID range: {bounds['lo']:,} to {bounds['hi']:,}")

## Process batches

For each batch: select a range of author IDs, join with work embeddings,
compute unweighted mean via `aggregate`, and INSERT INTO the output table.
Checkpoints after each batch so we can resume if interrupted.

In [ ]:
import time
from datetime import datetime

def get_completed_count():
    return spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']

def process_batch(id_lo, id_hi):
    """Compute mean embeddings for authors with IDs in [id_lo, id_hi)."""
    spark.sql(f"""
        INSERT INTO {OUTPUT_TABLE}
        WITH author_work_embeddings AS (
            SELECT
                CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
                e.embedding
            FROM {EMBEDDINGS_TABLE} e
            JOIN {WORKS_TABLE} w ON e.work_id = CAST(w.id AS STRING)
            LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
            WHERE a.author.id IS NOT NULL
              AND e.embedding IS NOT NULL
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) >= {id_lo}
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) < {id_hi}
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) NOT IN (
                  SELECT author_id FROM {OUTPUT_TABLE}
              )
        ),
        summed AS (
            SELECT
                author_id,
                CAST(COUNT(*) AS INT) AS work_count,
                aggregate(
                    collect_list(embedding),
                    cast(array_repeat(cast(0.0 AS DOUBLE), {EMBEDDING_DIM}) AS ARRAY<DOUBLE>),
                    (acc, x) -> transform(acc, (v, i) -> v + x[i])
                ) AS sum_embedding
            FROM author_work_embeddings
            GROUP BY author_id
        )
        SELECT
            author_id,
            work_count,
            transform(sum_embedding, v -> v / work_count) AS embedding
        FROM summed
    """)


# Build batch ranges
remaining_ids = spark.sql("""
    SELECT author_id FROM remaining_authors ORDER BY author_id
""").collect()

if not remaining_ids:
    print("No authors to process.")
else:
    all_ids = [r['author_id'] for r in remaining_ids]
    batches = []
    for i in range(0, len(all_ids), BATCH_SIZE):
        batch_ids = all_ids[i:i + BATCH_SIZE]
        batches.append((batch_ids[0], batch_ids[-1] + 1))  # [lo, hi)

    total_authors = len(all_ids)
    start_time = time.time()
    start_count = get_completed_count()

    print(f"Processing {total_authors:,} authors in {len(batches)} batches of ~{BATCH_SIZE:,}")
    print("=" * 70)

    for batch_num, (id_lo, id_hi) in enumerate(batches, 1):
        batch_start = time.time()
        try:
            process_batch(id_lo, id_hi)
            batch_elapsed = time.time() - batch_start

            current = get_completed_count()
            added = current - start_count
            total_elapsed = time.time() - start_time
            rate = added / total_elapsed if total_elapsed > 0 else 0
            remaining_n = total_authors - added
            eta_min = remaining_n / rate / 60 if rate > 0 else 0

            print(
                f"[{datetime.now().strftime('%H:%M:%S')}] "
                f"Batch {batch_num}/{len(batches)}: {batch_elapsed:.0f}s | "
                f"Total: {current:,} | "
                f"+{added:,} this run | "
                f"Rate: {rate:.0f}/s | "
                f"ETA: {eta_min:.0f}min"
            )
        except Exception as e:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] ERROR batch {batch_num} "
                  f"(IDs {id_lo}-{id_hi}): {e}")
            print("Waiting 60s before continuing...")
            time.sleep(60)

    print("=" * 70)
    final = get_completed_count()
    total_time = (time.time() - start_time) / 60
    print(f"Done! {final:,} total authors in {total_time:.1f}min")

## Verify results

In [ ]:
spark.sql(f"""
    SELECT
        COUNT(*) AS total_authors,
        AVG(work_count) AS avg_works_per_author,
        PERCENTILE(work_count, 0.5) AS median_works,
        PERCENTILE(work_count, 0.95) AS p95_works,
        MAX(work_count) AS max_works,
        SUM(CASE WHEN work_count = 1 THEN 1 ELSE 0 END) AS single_work_authors
    FROM {OUTPUT_TABLE}
""").show(truncate=False)

In [ ]:
# Verify all embeddings are 1536-dim
spark.sql(f"""
    SELECT SIZE(embedding) AS embedding_dim, COUNT(*) AS n
    FROM {OUTPUT_TABLE}
    GROUP BY SIZE(embedding)
""").show()

In [ ]:
# Spot check: top authors by work count with L2 norm
spark.sql(f"""
    SELECT
        author_id,
        work_count,
        ROUND(SQRT(AGGREGATE(embedding, CAST(0.0 AS DOUBLE), (acc, v) -> acc + v * v)), 4) AS l2_norm
    FROM {OUTPUT_TABLE}
    ORDER BY work_count DESC
    LIMIT 10
""").show(truncate=False)

## Optimize table

In [ ]:
spark.sql(f"OPTIMIZE {OUTPUT_TABLE}")
print(f"Table optimized: {OUTPUT_TABLE}")